# Zadanie 1: Predykcja jakości białego wina

In [54]:
import numpy as np
import pandas as pd

from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score,cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import optuna


In [32]:
data = pd.read_csv("../Data/winequality-white.csv",sep=';')
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [33]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [34]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


## Data Preparation

In [35]:
X = data.drop(['quality'],axis=1)
y = data['quality']

y.value_counts()

quality
6    2198
5    1457
7     880
8     175
4     163
3      20
9       5
Name: count, dtype: int64

In [36]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.3,shuffle=False,random_state=42)

X_train.shape

(3428, 11)

In [37]:
y.value_counts()

quality
6    2198
5    1457
7     880
8     175
4     163
3      20
9       5
Name: count, dtype: int64

In [38]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [45]:
def objective(trial:optuna.trial.Trial,regressor_name:str):
    n_features = trial.suggest_int("n_features",5,X_train.shape[1])
    
    if regressor_name == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators",50,300)
        max_depth = trial.suggest_int("max_depth",3,20)
        min_samples_split = trial.suggest_int("min_samples_split",2,20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1,10)

        regressor = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
        )
        
    elif regressor_name == "KNN":
        n_neighbors = trial.suggest_int("n_neighbors",1,5)
        regressor = KNeighborsRegressor(
            n_neighbors=n_neighbors
        )
    else:
        return
    
    pipeline = Pipeline([
        ("StandardScaler",StandardScaler()),
        ("SelectKBest", SelectKBest(k=n_features,score_func=f_regression)),
        ("Regressor",regressor)
    ])

    negative_mse = cross_val_score(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    return -negative_mse.mean()

In [46]:
study_random_forest = optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler())
study_random_forest.optimize(lambda trial: objective(trial,regressor_name="RandomForest"),n_trials=30)

[I 2025-11-24 18:06:40,285] A new study created in memory with name: no-name-d10523ad-141a-43ba-8c60-2f47b21ec43c
[I 2025-11-24 18:06:45,108] Trial 0 finished with value: 0.5148786358801443 and parameters: {'n_features': 6, 'n_estimators': 165, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.5148786358801443.
[I 2025-11-24 18:06:48,053] Trial 1 finished with value: 0.5271935074689517 and parameters: {'n_features': 6, 'n_estimators': 91, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.5148786358801443.
[I 2025-11-24 18:06:55,101] Trial 2 finished with value: 0.5355347766169303 and parameters: {'n_features': 6, 'n_estimators': 199, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.5148786358801443.
[I 2025-11-24 18:07:01,134] Trial 3 finished with value: 0.4329902418964706 and parameters: {'n_features': 11, 'n_estimators': 128, 'max_depth': 13, 'min_sam

In [47]:
study_knn = optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler())
study_knn.optimize(lambda trial: objective(trial,"KNN"),n_trials=30)

[I 2025-11-24 18:10:52,988] A new study created in memory with name: no-name-f0763650-95c4-4bf9-876f-ec7c0571b29c
[I 2025-11-24 18:10:53,051] Trial 0 finished with value: 0.6226420750782065 and parameters: {'n_features': 6, 'n_neighbors': 4}. Best is trial 0 with value: 0.6226420750782065.
[I 2025-11-24 18:10:53,114] Trial 1 finished with value: 0.5998893139111745 and parameters: {'n_features': 7, 'n_neighbors': 4}. Best is trial 1 with value: 0.5998893139111745.
[I 2025-11-24 18:10:53,166] Trial 2 finished with value: 0.5923823540678002 and parameters: {'n_features': 7, 'n_neighbors': 5}. Best is trial 2 with value: 0.5923823540678002.
[I 2025-11-24 18:10:53,228] Trial 3 finished with value: 0.5791087484837523 and parameters: {'n_features': 9, 'n_neighbors': 5}. Best is trial 3 with value: 0.5791087484837523.
[I 2025-11-24 18:10:53,300] Trial 4 finished with value: 0.5724714839011725 and parameters: {'n_features': 10, 'n_neighbors': 4}. Best is trial 4 with value: 0.5724714839011725.


In [50]:
study_random_forest.best_params

{'n_features': 11,
 'n_estimators': 180,
 'max_depth': 17,
 'min_samples_split': 4,
 'min_samples_leaf': 1}

In [51]:
study_knn.best_params

{'n_features': 11, 'n_neighbors': 5}

In [59]:
print(f"RandomForest: {study_random_forest.best_value}")
print(f"KNN: {study_knn.best_value}")

if study_random_forest.best_value < study_knn.best_value:
    print("Random Forest wygrał")
else:
    print("KNN wygrał")

RandomForest: 0.41084110587677153
KNN: 0.5582893234874764
Random Forest wygrał


In [55]:
def get_scores(regressor_name,best_params):
    n_features = best_params['n_features']

    if regressor_name == "RandomForest":
        regressor = RandomForestRegressor(
            n_estimators=best_params['n_estimators'],
            max_depth=best_params['max_depth'],
            min_samples_split=best_params['min_samples_split'],
            min_samples_leaf=best_params['min_samples_leaf'],
            n_jobs=-1
        )
    elif regressor_name == "KNN":
        regressor = KNeighborsRegressor(
            n_neighbors=best_params['n_neighbors']
        )
    
    pipeline = Pipeline([
        ("StandardScaler",StandardScaler()),
        ("SelectKBest",SelectKBest(score_func=f_regression,k=n_features)),
        ('Regressor',regressor)
    ])

    scores = cross_validate(pipeline,X_train,y_train,cv=cv,scoring='neg_mean_squared_error')

    return -scores['test_score']

In [56]:
mse_rf = get_scores("RandomForest",study_random_forest.best_params)
mse_knn = get_scores("KNN",study_knn.best_params)

In [57]:
from scipy.stats import wilcoxon

In [60]:
stat,p_value = wilcoxon(mse_rf,mse_knn)

print(f"Różnica w wynikach wynosi: {abs(study_random_forest.best_value - study_knn.best_value)}")
print(f"p-value: {p_value}")

if p_value < 0.05:
    print("Różnica jest znacząca")
else:
    print("Nie ma różnicy")


Różnica w wynikach wynosi: 0.14744821761070487
p-value: 0.0625
Nie ma różnicy


In [63]:
best_rf_model = Pipeline([
    ("StandardScaler", StandardScaler()),
    ("SelectKBest", SelectKBest(k=study_random_forest.best_params["n_features"], score_func=f_regression)),
    ("Regressor", RandomForestRegressor(
        n_estimators=study_random_forest.best_params["n_estimators"],
        max_depth=study_random_forest.best_params["max_depth"],
        min_samples_split=study_random_forest.best_params["min_samples_split"],
        min_samples_leaf=study_random_forest.best_params["min_samples_leaf"]
    ))
])

In [65]:
best_rf_model.fit(X_train,y_train)

,steps,"[('StandardScaler', ...), ('SelectKBest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,score_func,<function f_r...00237D7224B80>
,k,11
,n_estimators,180
,criterion,'squared_error'


In [66]:
y_pred = best_rf_model.predict(X_test)

In [67]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R²:", r2)

R²: 0.2967424775473303
